# React — `useState` & state management

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> Topic 8 gave you complete, correct event wiring with nothing to connect it to. This is the
> thing it connects to. The playground experiment is `08-state.jsx`.

## LESSON 25 — `useState`: a component that remembers

Every handler you wrote in topic 8 could log, calculate and call — but not change what was
on screen. Here is why, and what fixes it.

### Why a normal variable cannot work

```jsx
function Counter() {
  let count = 0;                               // a normal variable

  function handleClick() {
    count = count + 1;                         // this really does run
  }

  return <button onClick={handleClick}>{count}</button>;
}
```

Click it and nothing happens. React's docs give two separate reasons, and both are true at
once:

> **Local variables don't persist between renders.** When React renders this component a
> second time, it renders it from scratch — it doesn't consider any changes to the local
> variables.

> **Changes to local variables won't trigger renders.** React doesn't realize it needs to
> render the component again with the new data.

A component is a function, and every render calls it again. `let count = 0` runs again too,
so whatever you did to it last time is gone. And nothing told React to call it again in the
first place.

So two things have to happen, and React names them:

> 1. **Retain** the data between renders.
> 2. **Trigger** React to render the component with new data.

### `useState` does exactly those two

```jsx
import { useState } from "react";

function Counter() {
  const [count, setCount] = useState(0);

  function handleClick() {
    setCount(count + 1);
  }

  return <button onClick={handleClick}>{count}</button>;
}
```

`useState` returns **an array with exactly two values**, which you destructure:

| | |
|---|---|
| `count` | the current state. On the first render it is the initial value you passed |
| `setCount` | the **set function**, which updates the state and tells React to render again |

The naming convention is `[thing, setThing]`.

The `0` is only used once. React "saves the initial state once and ignores it on the next
renders" — on every later render `useState(0)` hands back whatever the state actually is,
not `0`.

### Why a setter instead of assigning

`count` is a `const`, so `count = count + 1` is a `TypeError` — but that is the least of it.
Even as a `let` it would fail for both reasons above: nothing would survive, and nothing
would tell React anything.

`setCount(next)` does the two jobs at once: it stores the next value **and** asks React to
render this component again.

### The cycle

```text
click
  -> your handler runs
  -> setCount(next) stores the value and schedules a render
  -> React calls your component again
  -> useState now returns the new value
  -> your JSX is rebuilt, and React updates the DOM where it differs
```

Two details worth having straight now, because they explain things you will otherwise trip
over:

- **The setter does not change the variable in the code that is already running.** React's
  docs: calling it "only affects what `useState` will return starting from the *next*
  render." Log `count` immediately after `setCount(...)` and you will still see the old
  value. That is not a bug, and LESSON 26 is about it.
- **A render does not guarantee a visible change.** If the new value is the same as the
  current one (compared with `Object.is`), React "will skip re-rendering the component and
  its children". And when it does render, it updates only the parts of the DOM that differ.

### State belongs to one component instance

Render the same component twice and you get two independent states. React's words:

> State is local to a component instance on the screen. In other words, if you render the
> same component twice, each copy will have completely isolated state!

And it only goes one way:

> Unlike props, **state is fully private to the component declaring it.** The parent
> component can't change it.

Props come **down** from a parent; state lives **inside** and belongs to that one copy.

> In development, `<StrictMode>` renders components an extra time (LESSON 3), so anything you
> log during render appears twice. It changes nothing about how state works.

### Key Notes

- A local variable fails twice over: it does not survive a render, and changing it triggers
  nothing.
- `useState(initial)` returns `[value, setter]`; the initial value is used only on the first
  render.
- The setter stores the next value **and** schedules a render. It does not change the value
  in the code that is already running.
- State belongs to one component instance and is private to it.

### Example

**Runnable — plain JS.** Not a model of React — just the plain fact underneath the first of
React's two reasons. A component is a function, and a function's local variables start again
on every call.

In [ ]:
// Pretend this function is a component, and calling it is a render.
function l25renderWithLocal() {
  let count = 0;       // runs again on every call
  count = count + 1;   // "the click"
  return count;
}

console.log("three renders:", l25renderWithLocal(), l25renderWithLocal(), l25renderWithLocal());

// Now something outside the function remembers, the way React does for you.
let l25remembered = 0;

function l25renderWithMemory() {
  l25remembered = l25remembered + 1;
  return l25remembered;
}

console.log("three renders:", l25renderWithMemory(), l25renderWithMemory(), l25renderWithMemory());

The second version keeps its value because the value does not live inside the function.
That is the half `useState` gives you for free — and React adds the half a plain variable can
never have: telling React to call the function again.

### Exercise

**Part 1 — in the notebook.** Predict each line before you run it, then check.

Write `l25counter()` which, each time it is called, returns the number of times it has been
called so far — `1`, `2`, `3` — using a variable that lives **outside** the function. Then
write `l25broken()` which tries to do the same with a variable **inside** the function, and
log both five times so the difference is on screen.

In a comment: which of React's two reasons does `l25broken` demonstrate, and which one can
plain JavaScript not demonstrate at all?

**Part 2 — in the playground.** Point `playground/src/App.jsx` at `./experiments/08-state.jsx`
and run it. You are going to add state yourself.

1. Add a **Reset** button to `Counter` that puts `count` back to `0`. It needs no new
   concepts — the setter takes any value.
2. Add a **second** state variable to `Counter`, called `clicks`, which counts every click on
   *either* of the counter's buttons — so pressing "+1 (state)" three times and "Reset" once
   leaves `clicks` at 4. Display it.
3. Click around in copy **A** only, then look at copy **B**. Write down what B's two numbers
   are, and why.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

Three things are easy to blur. For each row, say what it is and what it can do:

| | |
|---|---|
| `let count = 0` inside the component | ? |
| `count` from `useState(0)` | ? |
| `setCount` | ? |

Then answer in comments:

1. A component has `const [count, setCount] = useState(0)`. A colleague writes
   `count = count + 1` inside the handler and is surprised by the error. What is the error,
   and what would still be wrong if `count` had somehow been a `let`?
2. `setCount(0)` is called on a counter that is already `0`. What does React do, and will
   anything on the screen change?
3. The same `<Counter />` is rendered twice on a page. Clicking one does not move the other.
   Explain why in one sentence, using the word *instance*.
4. A parent wants to reset a child's counter. Given what this lesson said about who owns
   state, why can the parent not simply set it — and which lesson's idea would you reach for
   to let the parent ask? (Name the mechanism, not the implementation.)

In [ ]:
// Your code here

## LESSON 26 — The updater function, batching, and the value that will not change

LESSON 25 ended with one sentence you were told to park: the setter does not change the
state variable in the code that is already running. This lesson is that sentence, and the
two surprises that come out of it. Both have the same cause.

### Each render's values are fixed

When React calls your component, the value `useState` hands back belongs to **that render**.
React's words: *each render's state values are fixed*. A handler created during that render
closes over that value, and calling a setter does not reach back and change it.

Measured, from the playground experiment:

```text
handler start — count is 0
handler end   — count is still 0
```

Three setter calls happened between those two lines. `count` is still `0`, because `count`
is this render's value and always will be.

### So three identical calls collapse into one

```js
function handleClick() {
  setCount(count + 1);   // count is 0 -> "make it 1"
  setCount(count + 1);   // count is STILL 0 -> "make it 1"
  setCount(count + 1);   // still "make it 1"
}
```

The counter goes from `0` to **`1`**, not to `3`. Each line computed `0 + 1` because each
line read the same fixed `count`.

### Batching

The three calls do not cause three renders either. React's rule:

> React waits until *all* code in the event handlers has run before processing your state
> updates.

In the experiment the render line appears only **after** `handler end` — one render for the
whole handler, no matter how many setters it contained. That is batching, and it is why the
screen never flickers through intermediate values.

### The updater function

When the next value depends on the previous one, hand the setter a **function** instead of a
value:

```js
setCount((c) => c + 1);
```

React calls this an **updater function**. It queues the function, and

> During the next render, React goes through the queue and gives you the final updated state.

Each updater receives the result of the one before it, so three of them really do add three:

```js
setCount((c) => c + 1);   // 0 -> 1
setCount((c) => c + 1);   // 1 -> 2
setCount((c) => c + 1);   // 2 -> 3
```

Measured: the counter lands on **`3`**.

Note what did *not* change: `count` inside the handler is still `0` after all three calls.
The updater form fixes the **result**, not the reading.

### Which form to use

| the next value… | use |
|---|---|
| depends on the previous value (`+ 1`, toggling, appending) | `setCount((c) => c + 1)` |
| does not (a reset, a constant, a value from an event) | `setCount(0)` — plain and clearer |

`setCount(0)` gains nothing from an updater. Reach for the function form when you are
*calculating from what was there*, not as a reflex.

### Key Notes

- A render's state value is fixed; the setter cannot change it inside the running handler.
- Several `setCount(count + 1)` calls in one handler all compute from the same value and
  collapse into one step.
- React batches: it processes updates after the whole handler has run, then renders once.
- Use an updater function when the next value is calculated from the previous one.

### Example

**Runnable — plain JS.** **Conceptual model — not React's actual implementation.**

React's documented rule is simple enough to write down: a queued **value** replaces the
state, and a queued **function** is called with what is there so far. This cell is that rule,
nothing more — it is not how React stores or schedules anything.

In [ ]:
// Conceptual model — not React's actual implementation.
function l26applyQueue(start, queue) {
  let value = start;
  for (const update of queue) {
    value = typeof update === "function" ? update(value) : update;
  }
  return value;
}

// Three plain values, all computed from the same fixed count of 0:
console.log("three plain values: ", l26applyQueue(0, [0 + 1, 0 + 1, 0 + 1]));

// Three updater functions, each fed the previous result:
console.log("three updaters:     ", l26applyQueue(0, [(c) => c + 1, (c) => c + 1, (c) => c + 1]));

### Exercise

**Part 1.** Using `l26applyQueue` from the example, predict then check what each of these
leaves the state as, starting from `0`:

```js
[5]
[5, (c) => c + 1]
[(c) => c + 1, 5]
[(c) => c * 2, (c) => c + 3]
```

Log each result with the queue beside it. One of them shows why order matters more than you
might expect — say which, in a comment, and why.

**Part 2.** A component has `count` at `10`. Write, as two separate arrays, the queue that
each of these handlers would produce:

```js
function handleA() {
  setCount(count + 5);
  setCount(count + 5);
}

function handleB() {
  setCount((c) => c + 5);
  setCount((c) => c + 5);
}
```

Run both through `l26applyQueue` starting from `10` and log the two results. Then answer in
a comment: what is `count` inside each handler *after* the last line runs?

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Part 1 — in the playground.** Point `playground/src/App.jsx` at
`./experiments/09-updates.jsx` and run it with the console open.

1. Click **three × setCount(count + 1)**. What does the counter show, and what do the two
   handler lines say?
2. Note *where* the render lines appear relative to `handler end`. What does that tell you
   about when React does the work?
3. Reset, then click **three × setCount(c => c + 1)**. What does the counter show now?
4. Each render prints two lines rather than one. Why — and does it affect any of the numbers
   above?

**Part 2 — judgement.** For each, say plain value or updater function, in a few words:

1. A "clear filters" button that sets the count back to `0`.
2. A "+1" button on a like counter.
3. A button that sets the count to the number of items in a list it was handed as a prop.
4. A double-click handler that needs to add `2` by calling a `+1` helper twice.

In [ ]:
// Your code here

## LESSON 27 — Objects in state: replace, never edit

State is very often an object — a profile, a set of filters, a form's values. Everything
LESSON 25 and LESSON 26 said still applies. What is new is a single rule, and it catches
almost everyone once.

### The rule

> You should **treat any JavaScript object that you put into state as read-only.**

React puts it plainly: although objects in state are *technically* mutable, treat them as if
they were immutable — like numbers, booleans and strings — and **replace** them instead of
changing them.

### Why — and it is the LESSON 25 reason again

```js
profile.city = "Bologna";   // the object's contents changed
```

`profile` is still **the same object**. Nothing new was created. That is what breaks, and it
breaks in two different ways:

- If you only edit the field and never call the setter, nothing is scheduled at all. React
  was never told anything happened.
- If you edit the field and then call `setProfile(profile)`, React compares the new value
  with the current one using `Object.is` (LESSON 25), sees **the identical object**, and
  skips the re-render.

There is a nastier third case. If something *else* causes a render later, your edited value
appears on screen then — so the bug looks intermittent, and the click that "did nothing"
looks unrelated. Replacing the object instead of editing it removes the whole class of
problem.

You have met this identity point before: LESSON 21 showed that `.sort()` changes the array
it is given and hands back *the same array*. Same idea, now about state.

### Replace it with a copy

```js
setProfile({ ...profile, city: "Bologna" });
```

The spread copies every existing field; the key written afterwards wins. Fields you did not
name are preserved without being listed — which is the point, because listing them is how
they get quietly lost.

You already know this syntax from LESSON 9, where you spread an object into JSX attributes.
It is the same `...`.

### The copy is shallow

React's words:

> Note that the `...` spread syntax is "shallow" — it only copies things one level deep.

So an object *inside* your object is not copied. The copy and the original point at the very
same inner object, and editing it through one is visible through the other. To update a
nested field, spread at every level you are passing through:

```js
setProfile({
  ...profile,
  address: { ...profile.address, city: "Bologna" },
});
```

Deeply nested state is a sign the state should be shaped more flatly — but that is a design
question, and the mechanism above is what you need first.

### Building a new object is not mutation

The rule is about objects *already in state*. React is explicit:

> Mutating an object you've just created is okay because *no other code references it yet.*

So this is completely fine:

```js
const next = { ...profile };
next.city = "Bologna";
setProfile(next);
```

Nothing is watching `next`. Filling in an object you are about to hand over is ordinary
JavaScript, not a bug.

### With the updater form

The next object is calculated from the previous one, so LESSON 26 applies:

```js
setProfile((p) => ({ ...p, city: "Bologna" }));
```

Note the **parentheses around the object literal**. Without them, `(p) => { ... }` is read as
a function body, not an object, and the function returns `undefined`.

### Key Notes

- Treat an object in state as read-only: **replace** it, never edit it.
- Editing a field leaves the same object, so `Object.is` sees no change and React may not
  re-render — sometimes not until something unrelated does.
- `{ ...obj, key: value }` copies and overrides. The copy is **shallow**, so a nested object
  needs its own spread.
- An object you have just built is yours to fill in; the rule only covers what is already in
  state.

> *Further reading, not used in this course:* React's docs mention **Immer**, a library that
> lets you write mutating-looking code and produces the copies for you. Worth knowing it
> exists; plain spreads are what you should be fluent in first.

### Example

**Runnable — plain JS.** Nothing here imitates React. This is the ordinary object behaviour
that the rule exists because of — and identity is the thing to watch, not the values.

In [ ]:
const l27original = {
  name: "Ada",
  role: "admin",
  theme: { mode: "dark", accent: "teal" },
};

// 1 - editing a field. The contents changed; the object did not.
const l27edited = l27original;
l27edited.role = "editor";
console.log("1 same object?      ", Object.is(l27original, l27edited), "| role:", l27original.role);

// 2 - replacing. A genuinely new object, and the fields you did not name survive.
const l27replaced = { ...l27original, role: "viewer" };
console.log("2 same object?      ", Object.is(l27original, l27replaced), "| name:", l27replaced.name);

// 3 - but the copy is SHALLOW: both objects share one theme.
console.log("3 same theme?       ", Object.is(l27original.theme, l27replaced.theme));
l27replaced.theme.mode = "light";
console.log("3 original's mode:  ", l27original.theme.mode, "<- changed through the copy");

// 4 - spreading at every level you pass through.
const l27deep = { ...l27original, theme: { ...l27original.theme, accent: "rose" } };
console.log("4 same theme?       ", Object.is(l27original.theme, l27deep.theme));
console.log("4 original's accent:", l27original.theme.accent, "| copy's:", l27deep.theme.accent);

Line 1 is the bug: React would compare `l27original` with `l27edited`, find the same object,
and do nothing. Line 3 is the same bug one level down — which is why it is worth seeing
before you meet it.

### Exercise

**Part 1 — in the notebook.** Start from this settings object:

```js
const l27settings = {
  city: "Milan",
  notifications: true,
  display: { fontSize: 14, compact: false },
};
```

Write three functions. Each takes the settings object and **returns a new one**, leaving the
original completely untouched:

1. `l27withCity(settings, city)`
2. `l27toggleNotifications(settings)` — flips the boolean, preserving everything else
3. `l27withFontSize(settings, size)` — `fontSize` lives inside `display`

Then prove all three with `Object.is`: the result must not be the original, and after calling
`l27withFontSize` the original's `display.fontSize` must still be `14`. Log the checks, don't
just trust them.

**Part 2 — read and rewrite.** This handler is broken. It is not runnable here; answer in
comments.

```js
function handleUpgrade() {
  account.plan = "pro";
  setAccount(account);
}
```

Say what the user sees when they click, and why. Then write the corrected handler using the
updater form from LESSON 26, and say in one line why the updater form is the right choice
here.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

Answer in comments.

1. Is this a mutation bug?

```js
const next = { ...profile };
next.name = "Ada";
setProfile(next);
```

2. `profile` is `{ name: "Ada", role: "admin", city: "Milan" }` and a handler calls
   `setProfile({ city: "Bologna" })`. What is in state afterwards, and what did the user
   lose?

3. A colleague edits a field directly and says "it works — the screen updates". What is the
   most likely explanation, and why is it worse than a change that never appears at all?

4. State is `{ a: { n: 1 }, b: { n: 2 } }` and you update `a.n` by spreading at both levels.
   Is `b` copied? Is that a problem?

In [ ]:
// Your code here

## LESSON 28 — Arrays in state: add, remove, toggle, edit

Almost every React screen you will build holds a list in state. This is the lesson to
actually drill, because these four operations are most of the work in Mini-project 2 and a
good deal of everything after it.

The rule is the one you already have. React's words:

> Like with objects, **you should treat arrays in React state as read-only.** This means that
> you shouldn't reassign items inside an array like `arr[0] = 'bird'`, and you also shouldn't
> use methods that mutate the array, such as `push()` and `pop()`.

An array is an object, so LESSON 27 already told you why: editing one leaves the same array,
`Object.is` sees no change, and React may not re-render.

### Which method to reach for

React's own table:

| | avoid (mutates the array) | prefer (returns a new array) |
|---|---|---|
| adding | `push`, `unshift` | `concat`, `[...arr]` spread |
| removing | `pop`, `shift`, `splice` | `filter`, `slice` |
| replacing | `splice`, `arr[i] = ...` assignment | `map` |
| sorting | `reverse`, `sort` | copy the array first |

The JavaScript course gave you the **copying twins** for that last row: `toSorted`,
`toReversed`, `toSpliced`, and `with` for a single index. Use them — `toSorted(compare)` is
`[...items].sort(compare)` with nothing to forget.

### The four operations

```js
// add - the spread decides where the new item goes
setTasks([...tasks, newTask]);        // append
setTasks([newTask, ...tasks]);        // prepend

// remove - filter keeps what you want, it does not delete what you don't
setTasks(tasks.filter((t) => t.id !== id));

// toggle - map replaces ONE item with a NEW object
setTasks(tasks.map((t) => (t.id === id ? { ...t, done: !t.done } : t)));

// edit - same shape, different field
setTasks(tasks.map((t) => (t.id === id ? { ...t, text: nextText } : t)));
```

Toggle and edit are the same move: `map` builds the new **array**, and the spread builds the
new **object**. You need both, and the next section is why.

### The trap: a copied array holds the same items

React is explicit:

> Copying is shallow — the new array will contain the same items as the original one. So if
> you modify an object inside the copied array, you are mutating the existing state.

```js
const next = [...tasks];
next[0].done = true;     // <- next[0] and tasks[0] are the SAME object
setTasks(next);
```

The array is new, so React *will* re-render — and the bug hides behind that. You have
quietly changed the old state too, and nothing that compares before with after can tell the
truth any more.

The rule React gives is the one to memorise:

> You need to create copies from the point of change all the way to the top level.

`map` plus the object spread does exactly that, and nothing more. Items you did not touch are
returned as-is and stay the same objects — that is deliberate, and it is what keeps the
update cheap.

New items are the exception, as in LESSON 27: an object you have just built is yours to fill
in.

> Each item needs a stable `id` of its own — LESSON 20's key problem, now on the data side.
> In a component, a module-level counter or `crypto.randomUUID()` both work; the array index
> does not.

### Key Notes

- Treat an array in state as read-only, for the same reason as an object: `push`, `splice`,
  `sort` and `arr[i] = …` all leave you holding the same array.
- `[...arr, item]` adds · `filter` removes · `map` replaces one item · a copying twin sorts.
- Updating an object **inside** an array needs both copies: `map` for the array and `{ ...t }`
  for the object. Copy from the point of change up to the top.
- Items you did not change are returned unchanged, and sharing them is correct.

### Example

**Runnable — plain JS.** The four operations, then the trap. Watch the identity checks at the
end — those are the part that is easy to believe and get wrong.

In [ ]:
const l28tasks = [
  { id: "t1", text: "Write the brief", done: false },
  { id: "t2", text: "Review the copy", done: false },
];

// add
const l28added = [...l28tasks, { id: "t3", text: "Ship it", done: false }];
console.log("add     ->", l28added.map((t) => t.id).join(" "));

// remove
const l28removed = l28added.filter((t) => t.id !== "t2");
console.log("remove  ->", l28removed.map((t) => t.id).join(" "));

// toggle - map for the array, spread for the object
const l28toggled = l28tasks.map((t) => (t.id === "t1" ? { ...t, done: !t.done } : t));
console.log("toggle  ->", l28toggled[0].done, "| original still", l28tasks[0].done);

// edit - the same move, a different field
const l28edited = l28tasks.map((t) => (t.id === "t2" ? { ...t, text: "Review the headline" } : t));
console.log("edit    ->", l28edited[1].text, "| original still", l28tasks[1].text);

// what map actually produced
console.log("changed item is new?  ", !Object.is(l28tasks[0], l28toggled[0]));
console.log("untouched item shared?", Object.is(l28tasks[1], l28toggled[1]), "<- correct, and deliberate");

// the trap
const l28copy = [...l28tasks];
console.log("copy is a new array?  ", !Object.is(l28tasks, l28copy));
console.log("but item 0 is shared: ", Object.is(l28tasks[0], l28copy[0]), "<- editing it edits the original");

The last three lines are the whole lesson. A new array is not a new list of items, and
`map` is what gives you a new item exactly where you needed one.

### Exercise

**Part 1 — the drill.** This one is worth doing properly; you will write these four functions
many times in the projects ahead.

```js
const l28list = [
  { id: "a", text: "Draft the plan", done: false },
  { id: "b", text: "Book the room", done: true },
  { id: "c", text: "Send the invites", done: false },
];
```

Write four functions. Each takes the list and **returns a new one**, leaving the original
completely untouched:

1. `l28add(list, id, text)` — appends a new task with `done: false`
2. `l28remove(list, id)`
3. `l28toggle(list, id)` — flips that task's `done`
4. `l28rename(list, id, text)`

Then prove it, and log the proofs:

- the original still has 3 items after `l28add`
- `l28toggle(l28list, "a")` returns a **new** array whose item `"a"` is a **new object**
- in that same result, items `"b"` and `"c"` are **the same objects** as in the original
- `l28list` item `"a"` still has `done: false`

**Part 2 — read and fix.** Not runnable; answer in comments. Three handlers, three different
bugs:

```js
function addTask(text) {
  tasks.push({ id: nextId(), text, done: false });
  setTasks(tasks);
}

function sortByText() {
  setTasks(tasks.sort((a, b) => a.text.localeCompare(b.text)));
}

function markFirstDone() {
  const next = [...tasks];
  next[0].done = true;
  setTasks(next);
}
```

For each: say what actually happens on screen, and write the corrected line. The third is the
interesting one — say why it is worse than the other two even though the screen updates.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

Answer in comments.

1. Sort these into "safe for state" and "not safe", in one pass:
   `push` · `filter` · `splice` · `map` · `sort` · `concat` · `with` · `arr[i] = x` ·
   `toSorted`. What is the single question that separates the two groups?
2. A task list renders fine, but ticking a checkbox updates the row *and* silently changes
   what an "undo" button would restore. Which of the three bugs from Part 2 is this, and why
   does the screen look correct?
3. `map` returns untouched items as-is rather than copying them. Give one reason that is a
   feature rather than an oversight.
4. You need to add a task at position 2 rather than at either end. Using only what this
   lesson and LESSON 21 gave you, how would you build that array — and which mutating method
   are you deliberately not using?

In [ ]:
// Your code here

## LESSON 29 — Derived values are not state, and lifting state up

Two questions close this topic, and they are the same question asked from two sides:
**what deserves to be state, and where should it live?**

### Derived values are not state

LESSON 21 had you compute filtered lists and counts above the `return`, every render. Here is
the rule behind that habit, now that you have state to misuse:

> If you can calculate some information from the component's props or its existing state
> variables during rendering, you **should not** put that information into that component's
> state.

So this is wrong:

```js
const [tasks, setTasks] = useState(initialTasks);
const [remaining, setRemaining] = useState(initialTasks.length);   // redundant
```

and this is right:

```js
const [tasks, setTasks] = useState(initialTasks);
const remaining = tasks.filter((t) => !t.done).length;             // derived, every render
```

The reason is not tidiness. Two state variables holding the same fact have to be kept in
step **by hand, at every single place that changes either one**. Add a task and forget the
second setter, and the list says four while the badge says three. React's warning is blunt:

> When the same data is duplicated between multiple state variables, or within nested
> objects, it is difficult to keep them in sync.

A derived value cannot drift, because it does not exist between renders. It is recalculated
from the one thing that is true — and after LESSON 26 you know that a render is exactly when
that recalculation happens.

**The question to ask before adding a `useState`:** can I work this out from what I already
have? If yes, it is a `const` above the `return`, not state.

### Lifting state up

LESSON 25 left you with two counters that would not move together, and said the answer was
this lesson. Here it is.

> Sometimes, you want the state of two components to always change together. To do it,
> remove state from both of them, move it to their closest common parent, and then pass it
> down to them via props.

That is **lifting state up**, and React calls it one of the most common things you will do.
The three steps:

1. **Remove the state from the child components.**
2. **Pass the value down from the common parent** as a prop.
3. **Add the state to the common parent**, and pass an event handler down so the children can
   ask for a change.

**The React API.** The child stops owning the value and starts being told it:

```jsx
function Counter({ title, count, onIncrement }) {
  return (
    <section>
      <h2>{title}</h2>
      <p>{count}</p>
      <button onClick={onIncrement}>+1</button>
    </section>
  );
}

function Panel() {
  const [count, setCount] = useState(0);

  return (
    <>
      <Counter title="A" count={count} onIncrement={() => setCount((c) => c + 1)} />
      <Counter title="B" count={count} onIncrement={() => setCount((c) => c + 1)} />
    </>
  );
}
```

Nothing here is new. The value goes down as a prop (LESSON 14) and the request comes back up
as a function prop (LESSON 16). LESSON 25 told you a parent cannot reach into a child's state
— this is the answer it pointed at, and notice what actually changed: not the mechanism, but
**who owns the value**.

React's name for the principle:

> For each unique piece of state, you will choose the component that "owns" it. This
> principle is also known as having a "single source of truth".

### Where the state should live

Lifting is not a default. The rule that keeps applications simple is: **state lives in the
lowest component that still covers everyone who needs it.** Lift when two siblings must agree
or a parent must know; leave it where it is otherwise, because every lift adds props to
thread through.

And if the parent needs a *total* rather than a shared value, do not lift and duplicate —
lift the two counts and **derive** the total. That is the first half of this lesson doing the
work of the second.

> **A word you will meet:** a component holding its own state is called **uncontrolled**; one
> whose important values arrive as props is **controlled**. You have just converted `Counter`
> from the first to the second. This is about components in general — controlled *inputs* are
> topic 10 and are a specific case of it.

### Key Notes

- If you can calculate it from props or existing state during render, it is **not state** —
  it is a `const` above the `return`.
- Duplicated state has to be kept in sync by hand and eventually will not be. A derived value
  cannot drift.
- To make two components change together: remove the state from both, move it to the closest
  common parent, pass the value down and a handler back up.
- Keep state in the lowest component that covers everyone who needs it. Lift when you must,
  not by default.

### Example

**Runnable — plain JS.** The drift, measured. One operation is performed on a list, and the
two ways of knowing "how many are left" are compared afterwards.

In [ ]:
const l29tasks = [
  { id: "a", text: "Draft the plan", done: false },
  { id: "b", text: "Book the room", done: false },
  { id: "c", text: "Send the invites", done: true },
];

// Stored beside the list, the way a second useState would hold it.
let l29storedRemaining = l29tasks.filter((t) => !t.done).length;
console.log("start  -> stored:", l29storedRemaining, "| derived:", l29tasks.filter((t) => !t.done).length);

// An update (LESSON 28) that changes how many are left - and forgets the second setter.
const l29next = l29tasks.map((t) => (t.id === "a" ? { ...t, done: true } : t));

console.log("after  -> stored:", l29storedRemaining, "| derived:", l29next.filter((t) => !t.done).length);
console.log("agree? ", l29storedRemaining === l29next.filter((t) => !t.done).length);

// The stored number is not wrong because the code is bad. It is wrong because it is a
// SECOND copy of a fact, and the update only touched the first.

The derived line needed no maintenance and could not have been forgotten. That is the whole
argument — the stored one is one missed setter away from lying, permanently.

### Exercise

**Part 1 — in the notebook.** Using this list:

```js
const l29list = [
  { id: "a", text: "Draft the plan", done: false, hours: 3 },
  { id: "b", text: "Book the room", done: true, hours: 1 },
  { id: "c", text: "Send the invites", done: false, hours: 2 },
];
```

1. Write `l29summary(list)` returning an object with `total`, `done`, `remaining`,
   `hoursLeft` and `allDone` — every one of them calculated from the list, none stored.
2. Toggle `"a"` to done with your LESSON 28 move, then call `l29summary` on the result and log
   both summaries. Every number must agree with its list without you touching the summary.
3. In a comment, sort these into **state** and **derived**, for a task screen:
   the task list · the number of tasks left · the text typed into a "new task" box · whether
   the list is empty · which filter is selected ("all" / "done") · the visible tasks for that
   filter.

**Part 2 — in the playground.** Open `playground/src/experiments/08-state.jsx`. It renders two
`<Counter />` copies with independent state — that was LESSON 25's point, and now you are
going to change it.

Lift `count` into `Experiment08` so both copies show one shared number:

1. Delete `useState` from `Counter`; accept `count` and `onIncrement` as props instead.
2. Declare the state once, in `Experiment08`.
3. Pass `count` and a handler to both copies.
4. Click either button and confirm **both** numbers move.

Then, without writing it, answer in a comment: if each counter kept its **own** count and the
page also showed the total, what would you put in state and what would you derive?

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

Answer in comments.

1. A component has `const [items, setItems] = useState([])` and
   `const [isEmpty, setIsEmpty] = useState(true)`. Name the bug this will have before it is
   written, and give the one-line replacement.
2. Three sibling components need to read a value; only one of them changes it. Where does the
   state go, and what does each sibling receive?
3. A colleague lifts every piece of state to the top component "so everything is in one
   place". Give one concrete cost of that, in terms of what the code now has to carry.
4. LESSON 25 said a parent cannot change a child's state. Is that still true after lifting
   state up? Answer in one sentence that uses the word *owns*.

In [ ]:
// Your code here